# Day 2 - Lab 2: Documenting Key Decisions with ADRs

**Objective:** Use an LLM as a research assistant to compare technical options and synthesize the findings into a formal, version-controlled Architectural Decision Record (ADR).

**Estimated Time:** 60 minutes

**Introduction:**
Great architectural decisions are based on research and trade-offs. A critical practice for healthy, long-lived projects is documenting *why* these decisions were made. In this lab, you will use an LLM to research a key technical choice for our application and then generate a formal ADR to record that decision for the future.

For definitions of key terms used in this lab, please refer to the [GLOSSARY.md](../../GLOSSARY.md).

## Step 1: Setup

We'll start by ensuring our environment is ready and adding the standard pathing solution to reliably import our `utils.py` helper.

**Model Selection:**
For research and synthesis tasks, models with large context windows and strong reasoning abilities are ideal. `gpt-4.1`, `gemini-2.5-pro`, or `meta-llama/Llama-3.3-70B-Instruct` would be excellent choices.

**Helper Functions Used:**
- `setup_llm_client()`: To configure the API client.
- `get_completion()`: To send prompts to the LLM.
- `load_artifact()`: To read the ADR template.
- `save_artifact()`: To save the generated ADR template and the final ADR.

In [4]:
import sys
import os

# Add the project's root directory to the Python path to ensure 'utils' can be imported.
try:
    project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
except IndexError:
    project_root = os.path.abspath(os.path.join(os.getcwd()))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

from utils import setup_llm_client, get_completion, save_artifact, load_artifact

client, model_name, api_provider = setup_llm_client(model_name="gemini-2.5-pro")

2025-10-28 13:32:52,762 ag_aisoftdev.utils INFO LLM Client configured provider=google model=gemini-2.5-pro latency_ms=None artifacts_path=None


## Step 2: The Challenges

### Challenge 1 (Foundational): The ADR Template

**Task:** A good ADR follows a consistent format. Your first task is to prompt an LLM to generate a clean, reusable ADR template in markdown.

**Instructions:**
1.  Write a prompt that asks the LLM to generate a markdown template for an Architectural Decision Record.
2.  The template should include sections for: `Title`, `Status` (e.g., Proposed, Accepted, Deprecated), `Context` (the problem or forces at play), `Decision` (the chosen solution), and `Consequences` (the positive and negative results of the decision).
3.  Save the generated template to `templates/adr_template.md`.

In [6]:
# TODO: Write a prompt to generate a markdown ADR template.
adr_template_prompt = """ Create a markdown template for an Architectural Decision Record. 
The template should include sections for Title, Status (e.g. Proposed, Accepted, Deprecated),
Context (the problem or forces at play), Decision (the chosen solution), and Consequences 
(the positive and negative results of the decision). Your output should just be the markdown template
to be used directly by an automated pipeline."""

print("--- Generating ADR Template ---")
adr_template_content = get_completion(adr_template_prompt, client, model_name, api_provider)
print(adr_template_content)

# Save the artifact
if adr_template_content:
    save_artifact(adr_template_content, "templates/adr_template.md", overwrite=True)

--- Generating ADR Template ---
```markdown
# [ADR-000] - Title of the Architectural Decision

## Status

Proposed | Accepted | Deprecated | Superseded

## Context

[Describe the context and problem statement that this decision addresses. This section should cover the forces at play, including technological, political, social, and project constraints. It should be a clear and concise summary of the issue.]

## Decision

[This is the "what" of the decision. Describe the change that we are proposing or have agreed to. It should be a clear and concise statement of the chosen solution.]

## Consequences

[This section describes the "so what" of the decision. All consequences should be listed here, not just the positive ones. A decision without any negative consequences is a unicorn. This section should be a realistic assessment of the impact.]

### Positive

*   [List the positive outcomes of this decision. e.g., improved performance, easier maintenance, reduced cost, better alignment with

### Challenge 2 (Intermediate): AI-Assisted Research

**Task:** Use the LLM to perform unbiased research on a key technical decision for our project: choosing a database for semantic search.

**Instructions:**
1.  Write a prompt instructing the LLM to perform a technical comparison.
2.  Ask it to compare and contrast two technical options: **"Using PostgreSQL with the `pgvector` extension"** versus **"Using a specialized vector database like ChromaDB or FAISS"**.
3.  The prompt should ask for a balanced view for the specific use case of our new hire onboarding tool.
4.  Store the output in a variable for the next step.

> **Tip:** To get a balanced comparison, explicitly ask the LLM to 'act as an unbiased research assistant' and to list the 'pros and cons for each approach.' This prevents the model from simply recommending the more popular option and encourages a more critical analysis.

In [8]:
# TODO: Write a prompt to research database options.
db_research_prompt = """ You are an unbiased research assistant. Perform a report 
for a technical comparison to compare and contrast two technical options in the use case of an 
onboarding tool for new hires: 
    1. Using PostgreSQL with the pgvector extension 
    2. Using a specialized vector database like ChromaDB or FAISS

Your output should only include the relevant research.
"""

print("--- Researching Database Options ---")
db_research_output = get_completion(db_research_prompt, client, model_name, api_provider)
print(db_research_output)

--- Researching Database Options ---
### **Technical Comparison Report: Vector Storage for a New Hire Onboarding Tool**

**Objective:** This report provides a technical comparison between two distinct approaches for implementing semantic search capabilities within a new hire onboarding tool. The primary function is to allow new employees to ask natural language questions and receive relevant information from a knowledge base of internal documents (e.g., handbooks, policies, guides).

**Options Under Evaluation:**
1.  **PostgreSQL with the `pgvector` extension:** A general-purpose relational database augmented with vector similarity search capabilities.
2.  **Specialized Vector Database:** A purpose-built database optimized for storing and querying high-dimensional vector embeddings (e.g., ChromaDB, FAISS).

---

### **1. Option 1: PostgreSQL with `pgvector`**

#### **1.1. Technical Description**
PostgreSQL is a mature, open-source object-relational database system known for its reliabi

### Challenge 3 (Advanced): Synthesizing the ADR

**Task:** Provide the LLM with your research from the previous step and have it formally document the decision.

**Instructions:**
1.  Load the `adr_template.md` you created in the first challenge.
2.  Create a new prompt instructing the LLM to act as a Staff Engineer.
3.  Provide the `db_research_output` as context.
4.  Instruct the LLM to populate the ADR template, formally documenting the decision to **use PostgreSQL with pgvector** and justifying the choice based on the synthesized pros and cons.
5.  Save the final, completed ADR as `artifacts/adr_001_database_choice.md`.

In [11]:
adr_template = load_artifact("templates/adr_template.md")

# TODO: Write a prompt to synthesize the final ADR.
synthesis_prompt = f"""
You are a staff engineer, helping build a new onboarding tool for new hires,
that must formally document the decision to use *PosgreSQL with pgvector* by filling out the given
ADR template with the provided research pros and cons. 

<ADR Context>
{adr_template}
<ADR Context>

<Research Context>
{db_research_output}
<Research Context>

Your output should only be the filled out ADR markdown template that was provided.
"""

print("--- Synthesizing Final ADR ---")
if adr_template and 'db_research_output' in locals() and db_research_output:
    final_adr = get_completion(synthesis_prompt, client, model_name, api_provider)
    print(final_adr)
    save_artifact(final_adr, "artifacts/adr_001_database_choice.md", overwrite=True)
else:
    print("Skipping ADR synthesis because template or research is missing.")

--- Synthesizing Final ADR ---
# ADR-001 - Vector Storage and Search for Onboarding Tool

## Status

Accepted

## Context

The new hire onboarding tool requires a semantic search capability to allow new employees to ask natural language questions and receive relevant information from a knowledge base of internal documents (e.g., handbooks, policies, guides). This functionality necessitates storing vector embeddings of the document content and performing similarity searches on them.

We evaluated two primary architectural approaches:
1.  Using our existing relational database, PostgreSQL, augmented with the `pgvector` extension.
2.  Introducing a new, specialized vector database (e.g., ChromaDB) to run alongside our primary database.

The decision hinges on balancing query capabilities, operational complexity, data consistency, and performance for the specific scale and requirements of this onboarding tool. The key challenge is to enable powerful, metadata-aware searches (e.g., "find po

## Lab Conclusion

Well done! You have used an LLM to automate a complex but critical part of the architectural process. You leveraged its vast knowledge base for research and then used it again for synthesis, turning raw analysis into a formal, structured document. This `adr_001_database_choice.md` file now serves as a permanent, valuable record for anyone who works on this project in the future.

> **Key Takeaway:** The pattern of **Research -> Synthesize -> Format** is a powerful workflow. You can use an LLM to gather unstructured information and then use it again to pour that information into a structured template, creating high-quality, consistent documentation with minimal effort.